In [1]:
import folium
import numpy as np
import math
import pandas as pd
import os


In [2]:
r = 6378

def deg_to_rad(deg):
    return deg * np.pi / 180

def calcular_distance(lat1, lon1, lat2, lon2):
    d_lat = deg_to_rad(lat1 - lat2)
    d_lon = deg_to_rad(lon1 - lon2)
    a = np.sin(d_lat / 2) ** 2 + np.cos(deg_to_rad(lat1)) * np.cos(deg_to_rad(lat2)) * np.sin(d_lon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c

def es_valida(coordenada):
    try:
        if coordenada in [None, ""]:
            return False
        float(coordenada)
    except:
        return False


In [3]:
# Leer resultados ya calculados 
df = pd.read_csv("edificios_10km_o_mas.csv")

df["distancia_km"] = pd.to_numeric(df["distancia_km"], errors="coerce")

df_filtrado = df[df["distancia_km"] >= 10].copy()

for c in ["latitud", "longitud", "lat_cp", "lon_cp"]:
    df_filtrado[c] = pd.to_numeric(df_filtrado[c], errors="coerce")

print("Total filas CSV:", len(df))
print("Filas >= 10 km:", len(df_filtrado))

resultados = df_filtrado.to_dict(orient="records")


Total filas CSV: 4825
Filas >= 10 km: 4825


In [4]:
# Crear carpeta para mapas individuales
os.makedirs("mapas", exist_ok=True)

# lista con links
filas_index = []

# Agregar marcadores al mapa
for i, row in enumerate(resultados):
    lat_edificio, lon_edificio = row.get("latitud"), row.get("longitud")
    lat_cp, lon_cp = row.get("lat_cp"), row.get("lon_cp")
    distancia = row.get("distancia_km")
    cp = row.get("codigo_postal", "")

    # Solo brinca si NO se puede usar como número 
    if not all(map(es_valida, [lat_edificio, lon_edificio, lat_cp, lon_cp, distancia])):
        continue

    lat_edificio, lon_edificio, lat_cp, lon_cp = map(float, [lat_edificio, lon_edificio, lat_cp, lon_cp])
    distancia = float(distancia)

    # Zoom global para que se vea aunque quede lejos
    mapa = folium.Map(location=[lat_edificio, lon_edificio], zoom_start=2)

    folium.Marker(
        location=[lat_edificio, lon_edificio],
        popup=f"Edificio<br>CP: {cp}<br>Distancia: {distancia:.2f} km",
        icon=folium.Icon(color='blue')
    ).add_to(mapa)

    folium.Marker(
        location=[lat_cp, lon_cp],
        popup=f"CP/Colonia<br>CP: {cp}<br>Distancia: {distancia:.2f} km",
        icon=folium.Icon(color='red')
    ).add_to(mapa)

    folium.PolyLine(
        locations=[(lat_edificio, lon_edificio), (lat_cp, lon_cp)],
        color='green',
        tooltip=f"Distancia: {distancia:.2f} km"
    ).add_to(mapa)

    # Guardar mapa individual
    mapa_path = f"mapas/mapa_{i}.html"
    mapa.save(mapa_path)

    # Guardar fila para el index
    filas_index.append((i, cp, distancia, mapa_path))

with open("index.html", "w", encoding="utf-8") as f:
    f.write("<h2>Edificios con distancia ≥ 10 km</h2>")
    f.write("<p>Tecnología libre: Python + Folium (Leaflet) + OpenStreetMap</p>")
    f.write("<table border='1' cellpadding='6' cellspacing='0'>")
    f.write("<tr><th>#</th><th>Codigo Postal</th><th>Distancia (km)</th><th>Mapa</th></tr>")

    for i, cp, dist, mapa_path in filas_index:
        f.write(
            f"<tr>"
            f"<td>{i}</td>"
            f"<td>{cp}</td>"
            f"<td>{dist:.2f}</td>"
            f"<td><a href='{mapa_path}' target='_blank'>Ver y analizar</a></td>"
            f"</tr>"
        )

    f.write("</table>")

print("Listo. Abre 'index.html' en tu navegador.")
print("Mapas generados en la carpeta 'mapas/'.")


Listo. Abre 'index.html' en tu navegador.
Mapas generados en la carpeta 'mapas/'.
